In [1]:
import jax
import jax.numpy as jnp
from flax import nnx


class Linear(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.w = nnx.Param(jax.random.uniform(key, (din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout
        self.dropout = nnx.Dropout(0.5, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = x @ self.w + self.b
        return self.dropout(y)


class Linear2(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.w = nnx.Param(jax.random.uniform(key, (din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout
        self.dropout = nnx.Dropout(0.5, rngs=rngs)
        self.lin = Linear(din, dout, rngs=rngs)

    def __call__(self, x: jax.Array):
        y = x @ self.w + self.b
        return self.dropout(y)


model = Linear2(2, 5, rngs=nnx.Rngs(2, dropout=42))
y = model(x=jnp.ones((1, 2)))

print(y)
nnx.display(model)

graphdef, drop_state, other_state = nnx.split(model, nnx.All(nnx.PathContains("lin"), nnx.Param), ...)
nnx.state(model)

[[0.        2.8599195 0.        0.        0.       ]]


State({
  'b': VariableState( # 5 (20 B)
    type=Param,
    value=Array([0., 0., 0., 0., 0.], dtype=float32)
  ),
  'dropout': {
    'rngs': {
      'default': {
        'count': VariableState( # 1 (4 B)
          type=RngCount,
          value=Array(2, dtype=uint32),
          tag='default'
        ),
        'key': VariableState( # 1 (8 B)
          type=RngKey,
          value=Array((), dtype=key<fry>) overlaying:
          [0 2],
          tag='default'
        )
      },
      'dropout': {
        'count': VariableState( # 1 (4 B)
          type=RngCount,
          value=Array(1, dtype=uint32),
          tag='dropout'
        ),
        'key': VariableState( # 1 (8 B)
          type=RngKey,
          value=Array((), dtype=key<fry>) overlaying:
          [ 0 42],
          tag='dropout'
        )
      }
    }
  },
  'lin': {
    'b': VariableState( # 5 (20 B)
      type=Param,
      value=Array([0., 0., 0., 0., 0.], dtype=float32)
    ),
    'w': VariableState( # 10 (40 B)
      

In [2]:
drop_state

State({
  'lin': {
    'b': VariableState( # 5 (20 B)
      type=Param,
      value=Array([0., 0., 0., 0., 0.], dtype=float32)
    ),
    'w': VariableState( # 10 (40 B)
      type=Param,
      value=Array([[0.99568295, 0.5087098 , 0.01863182, 0.89239514, 0.29435277],
             [0.9759338 , 0.6530578 , 0.37484288, 0.9107307 , 0.20158172]],      dtype=float32)
    )
  }
})

In [4]:
rngs = nnx.Rngs(params=0, dropout=1)


@nnx.split_rngs(splits=5, only="dropout")
def f(rngs: nnx.Rngs):
    print("Inside:")
    # rngs.dropout() # ValueError: fold_in accepts a single key...
    nnx.display(rngs)


f(rngs)

print("Outside:")
rngs.dropout()  # works!
nnx.display(rngs)

Inside:


Outside:


### Test random policy

In [5]:
from hcnn.policy.policy_mlp import PolicyMLP

rngs = nnx.Rngs(42, dropout=1)
policy = PolicyMLP.factory([2, 3, 2], 0.2, rngs=rngs)

random_state = nnx.state(policy, nnx.RngState)  # returns value, not reference
nnx.display(random_state)

nnx.display(policy)

In [6]:
print(policy(jnp.array([1, 2])))
print(policy(jnp.array([1, 2])))

[0.21044265 0.36397767]
[-0.01291138 -1.4362088 ]


In [7]:
nnx.display(policy)

In [8]:
nnx.display(random_state)

In [9]:
nnx.update(policy, random_state)
print(policy(jnp.array([1, 1])))
print(policy(jnp.array([1, 1])))

[0.26891333 0.24681942]
[ 0.08261265 -1.254725  ]


In [21]:
class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        self.linear1 = Linear(din, dmid, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)
        self.linear2 = Linear(dmid, dout, rngs=rngs)

    def __call__(self, x: jax.Array, rngs=None):
        x = nnx.gelu(self.dropout(self.linear1(x), rngs=rngs))
        return self.linear2(x)


model = MLP(2, 16, 5, rngs=nnx.Rngs(0, dropout=42))

batch_num = 20
x = jnp.ones((20, 2))


@nnx.vmap(in_axes=(None, 0, None), out_axes=0)
def forward(model, x: jnp.ndarray, rngs):
    return model(x, rngs=rngs)


y = forward(policy, x, rngs=nnx.clone(policy.rngs))
y

Array([[ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ],
       [ 0.08261265, -1.254725  ]], dtype=float32)

In [30]:
tuple_list = [
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
    (jnp.array([1, 2, 3]), jnp.array([4, 5, 6])),
]

lhs_list, rhs_list = zip(*tuple_list, strict=False)

all_lhs = jnp.concatenate(lhs_list)
all_rhs = jnp.concatenate(rhs_list)

all_lhs

Array([1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3], dtype=int32)